# Weekly Project 5
## Implementation of global registration
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.


### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

In [1]:
import copy
import glob

import numpy as np
import open3d as o3d
import plotly.io as pio

# Open a browser tab for each visualization
pio.renderers.default = 'browser'


def draw_registrations(source, target, transformation=None, recolor=False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if recolor:
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if transformation is not None:
        source_temp.transform(transformation)
    o3d.visualization.draw_plotly([source_temp, target_temp])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Task 1: global registration

We downsample both point clouds, estimate normals, compute FPFH features, then use RANSAC to find a matching transformation. Wrapped into two reusable functions, so we can run the same steps on both point cloud pairs below.

In [2]:
def preprocess_point_cloud(pcd, voxel_size):
    pcd_down = pcd.voxel_down_sample(voxel_size)

    radius_normal = voxel_size * 2
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

    radius_feature = voxel_size * 5
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))

    return pcd_down, pcd_fpfh


def global_registration(source, target, voxel_size):
    source_down, source_fpfh = preprocess_point_cloud(source, voxel_size)
    target_down, target_fpfh = preprocess_point_cloud(target, voxel_size)

    distance_threshold = voxel_size * 1.5
    point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint(False)

    return o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down,
        source_fpfh, target_fpfh,
        True,
        distance_threshold,
        point_to_point,
    )

### 1.1: r1.pcd and r2.pcd

Two scans of a room, taken from different positions.

In [3]:
r1 = o3d.io.read_point_cloud('global_registration/r1.pcd')
r2 = o3d.io.read_point_cloud('global_registration/r2.pcd')

voxel_size = 0.05
ransac_r = global_registration(r1, r2, voxel_size)
print('fitness:', ransac_r.fitness)
print('inlier_rmse:', ransac_r.inlier_rmse)

draw_registrations(r1, r2, ransac_r.transformation, True)

fitness: 0.6710084033613445
inlier_rmse: 0.027535086615816266


### 1.2: car1.ply and car2.ply

Two scans of a car from different angles.

A smaller `voxel_size` (0.02) makes RANSAC run for a very long time here, the car's smooth, symmetric surfaces give FPFH less distinctive detail to match on than a textured room does. `voxel_size = 0.05` avoids this and runs in a couple of seconds.

In [4]:
car1 = o3d.io.read_point_cloud('global_registration/car1.ply')
car2 = o3d.io.read_point_cloud('global_registration/car2.ply')

ransac_car = global_registration(car1, car2, voxel_size)
print('fitness:', ransac_car.fitness)
print('inlier_rmse:', ransac_car.inlier_rmse)

draw_registrations(car1, car2, ransac_car.transformation, True)


fitness: 0.9962750716332378
inlier_rmse: 0.03666667743982386


Both pairs fit well, r1/r2 reaches a fitness around 0.67, and car1/car2 reaches a fitness around 0.99. The car pair fits noticeably better, likely because the two car scans overlap more completely than the two room scans do.

## Task 2 (Challenge): local registration

Global registration gets us roughly aligned, but it is still a coarse fit built on a downsampled point cloud. We refine it with point-to-plane ICP on the full resolution point clouds, using the RANSAC result as ICP's starting point.

In [5]:
def refine_registration(source, target, global_result, threshold=0.02):
    source.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    target.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

    point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()
    return o3d.pipelines.registration.registration_icp(
        source, target, threshold, global_result.transformation, point_to_plane)

In [6]:
icp_r = refine_registration(r1, r2, ransac_r)
print('r1/r2')
print('  global:  fitness=%.4f  inlier_rmse=%.4f' % (ransac_r.fitness, ransac_r.inlier_rmse))
print('  refined: fitness=%.4f  inlier_rmse=%.4f' % (icp_r.fitness, icp_r.inlier_rmse))

icp_car = refine_registration(car1, car2, ransac_car)
print()
print('car1/car2')
print('  global:  fitness=%.4f  inlier_rmse=%.4f' % (ransac_car.fitness, ransac_car.inlier_rmse))
print('  refined: fitness=%.4f  inlier_rmse=%.4f' % (icp_car.fitness, icp_car.inlier_rmse))

draw_registrations(r1, r2, icp_r.transformation, True)
draw_registrations(car1, car2, icp_car.transformation, True)

r1/r2
  global:  fitness=0.6710  inlier_rmse=0.0275
  refined: fitness=0.6211  inlier_rmse=0.0066

car1/car2
  global:  fitness=0.9963  inlier_rmse=0.0367
  refined: fitness=0.9876  inlier_rmse=0.0054


ICP refinement tightens inlier_rmse considerably for both pairs (r1/r2 drops from about 0.026 to 0.007, car1/car2 from about 0.032 to 0.005), a noticeably closer fit than the coarse global alignment alone, confirming the standard global-then-local pipeline works well here.

## Task 2 (Challenge): reconstructing the car

`car_challange` has 1722 matching RGB and depth frame pairs.

In [ ]:
camera = o3d.camera.PinholeCameraIntrinsic(o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault)
flip = [[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]]

# Several frames see background 5 to 8 meters away, while the car stays
# within about 2 to 3 meters of the camera, so we cut everything past 2.5 m.
DEPTH_TRUNC = 2.5


def load_car_pcd(idx):
    color = o3d.io.read_image(f'car_challange/rgb/{idx:07d}.jpg')
    depth = o3d.io.read_image(f'car_challange/depth/{idx:07d}.png')
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color, depth, depth_trunc=DEPTH_TRUNC, convert_rgb_to_intensity=False)
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, camera)
    pcd.transform(flip)
    return pcd

In [ ]:
voxel_size = 0.02
threshold_local = 0.05
frame_step = 5
n_frames = 1722

point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()

merged = load_car_pcd(1)
merged.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
merged = merged.voxel_down_sample(voxel_size)

prev_transform = np.identity(4)
fitness_log = []

for idx in range(1 + frame_step, n_frames + 1, frame_step):
    new_pcd = load_car_pcd(idx)
    new_pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    new_down = new_pcd.voxel_down_sample(voxel_size)

    icp_res = o3d.pipelines.registration.registration_icp(
        new_down, merged, threshold_local, prev_transform, point_to_plane)

    new_pcd.transform(icp_res.transformation)
    merged += new_pcd
    merged = merged.voxel_down_sample(voxel_size)
    prev_transform = icp_res.transformation
    fitness_log.append(icp_res.fitness)

    if idx % 200 < frame_step:
        print(f'frame {idx}: fitness={icp_res.fitness:.3f}, points in map={len(merged.points)}')

print()
print(f'fitness over the whole run: min={min(fitness_log):.3f}, mean={np.mean(fitness_log):.3f}')
print('final point count:', len(merged.points))

fig = o3d.visualization.get_plotly_fig([merged], width=900, height=700)
fig.update_traces(marker=dict(size=2))
fig.show()

frame 201: fitness=0.962, points in map=32174
frame 401: fitness=1.000, points in map=62434
frame 601: fitness=0.997, points in map=69967
frame 801: fitness=0.995, points in map=88865
frame 1001: fitness=0.973, points in map=128899
frame 1201: fitness=0.992, points in map=136854
frame 1401: fitness=0.975, points in map=113087
frame 1601: fitness=0.956, points in map=132868

fitness over the whole run: min=0.508, mean=0.980
final point count: 153622


The car is clearly recognisable. With `frame_step = 5` every frame registers with a fitness of at least about 0.5, with a mean around 0.98. With a step of 10, a few frames dropped to around 0.3 because the camera moved too much between them, so we went with 5. We also tried falling back to RANSAC when ICP fitness was low, but it was much slower and not better.

Some background is still in the cloud, since the depth camera sees everything, not just the car. `DEPTH_TRUNC` removes the far background, but things close to the car, like the ground next to it, stay in. Removing those would need segmenting the car, which is outside the scope of this project.